GTFS signifie General Transit Feed Specification.
C’est un format standard international utilisé par les réseaux de transport pour publier leurs données.

🚍 GTFS = un ensemble de fichiers décrivant un réseau de transport

Un fichier GTFS contient plusieurs tables (au format CSV/TXT) qui décrivent :

1. Les arrêts

stops.txt
→ localisation, nom, identifiant, etc.

2. Les lignes

routes.txt
→ lignes de bus, métro, tram, leurs codes, couleurs, etc.

3. Les trajets

trips.txt
→ un “trip” = un aller simple d’un véhicule pour une ligne donnée.

4. Les horaires

stop_times.txt
→ pour chaque trip : les heures de passage à chaque arrêt.

5. Les calendriers

calendar.txt / calendar_dates.txt
→ quels jours les services fonctionnent.

6. Les formes géographiques

shapes.txt
→ la géométrie du trajet (les courbes sur la carte).

In [ ]:
import pandas as pd

# Charger les fichiers GTFS
stops = pd.read_csv('stops.txt', dtype={'stop_id': str})
trips = pd.read_csv('trips.txt', dtype={'trip_id': str, 'route_id': str})
stop_times = pd.read_csv('stop_times.txt', dtype={'trip_id': str, 'stop_id': str})
calendar = pd.read_csv('calendar.txt', dtype={'sservice_id': str})

In [ ]:
# Fusionner stop_times avec trips pour avoir route_id et service_id
stop_times_trips = stop_times.merge(trips[['trip_id', 'route_id', 'service_id']], on='trip_id', how='left')

In [ ]:
# Fusionner avec stops pour avoir stop_name 
full_df = stop_times_trips.merge(stops[['stop_id', 'stop_name']], on='stop_id', how='left')

full_df.head()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint,route_id,service_id,stop_name
0,00000000001,05:16:00,05:16:00,6-5001,1,NaN,0,1,92.210907,1,6-1001,prog-00000000002,J.F. Kennedy
1,00000000001,05:16:59,05:16:59,6-5002,2,NaN,0,0,616.730286,1,6-1001,prog-00000000002,Villejean-Université
2,00000000001,05:18:29,05:18:29,6-5003,3,NaN,0,0,1414.565674,1,6-1001,prog-00000000002,Pontchaillou
3,00000000001,05:19:39,05:19:39,6-5004,4,NaN,0,0,2050.249023,1,6-1001,prog-00000000002,Anatole France
4,00000000001,05:20:47,05:20:47,6-5005,5,NaN,0,0,2726.148193,1,6-1001,prog-00000000002,Sainte-Anne


In [ ]:
# Convertir arrival_time en timedelta pour gérer >24h 
full_df['arrival_time'] = pd.to_timedelta(full_df['arrival_time'])
full_df['departure_time'] = pd.to_timedelta(full_df['departure_time'])

In [ ]:
# Déterminer si le service est semaine ou week-end 
# Créer une colonne "day_type" selon calendar.txt
# 1=lundi, 2=mardi ... 7=dimanche, on considère samedi/dimanche comme weekend
full_df = full_df.merge(calendar[['service_id','monday','tuesday','wednesday','thursday','friday','saturday','sunday']], on='service_id', how='left')

def day_type(row):
    if row['saturday']==1 or row['sunday']==1:
        return 'weekend'
    else:
        return 'weekday'

full_df['day_type'] = full_df.apply(day_type, axis=1)

full_df.head()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint,...,service_id,stop_name,monday,tuesday,wednesday,thursday,friday,saturday,sunday,day_type
0,00000000001,0 days 05:16:00,0 days 05:16:00,6-5001,1,NaN,0,1,92.210907,1,...,prog-00000000002,J.F. Kennedy,1,0,0,0,0,0,0,weekday
1,00000000001,0 days 05:16:59,0 days 05:16:59,6-5002,2,NaN,0,0,616.730286,1,...,prog-00000000002,Villejean-Université,1,0,0,0,0,0,0,weekday
2,00000000001,0 days 05:18:29,0 days 05:18:29,6-5003,3,NaN,0,0,1414.565674,1,...,prog-00000000002,Pontchaillou,1,0,0,0,0,0,0,weekday
3,00000000001,0 days 05:19:39,0 days 05:19:39,6-5004,4,NaN,0,0,2050.249023,1,...,prog-00000000002,Anatole France,1,0,0,0,0,0,0,weekday
4,00000000001,0 days 05:20:47,0 days 05:20:47,6-5005,5,NaN,0,0,2726.148193,1,...,prog-00000000002,Sainte-Anne,1,0,0,0,0,0,0,weekday


In [9]:
full_df

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint,...,service_id,stop_name,monday,tuesday,wednesday,thursday,friday,saturday,sunday,day_type
0,00000000001,0 days 05:16:00,0 days 05:16:00,6-5001,1,NaN,0,1,92.210907,1,...,prog-00000000002,J.F. Kennedy,1,0,0,0,0,0,0,weekday
1,00000000001,0 days 05:16:59,0 days 05:16:59,6-5002,2,NaN,0,0,616.730286,1,...,prog-00000000002,Villejean-Université,1,0,0,0,0,0,0,weekday
2,00000000001,0 days 05:18:29,0 days 05:18:29,6-5003,3,NaN,0,0,1414.565674,1,...,prog-00000000002,Pontchaillou,1,0,0,0,0,0,0,weekday
3,00000000001,0 days 05:19:39,0 days 05:19:39,6-5004,4,NaN,0,0,2050.249023,1,...,prog-00000000002,Anatole France,1,0,0,0,0,0,0,weekday
4,00000000001,0 days 05:20:47,0 days 05:20:47,6-5005,5,NaN,0,0,2726.148193,1,...,prog-00000000002,Sainte-Anne,1,0,0,0,0,0,0,weekday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1460546,12_SAMEDI-16735902,1 days 01:22:00,1 days 01:22:00,6-2161,4,NaN,0,0,NaN,1,...,expl-12_SAMEDI,Rigourdière,0,0,0,0,0,1,0,weekend
1460547,12_SAMEDI-16735903,1 days 01:20:00,1 days 01:20:00,6-1615,1,NaN,0,0,NaN,1,...,expl-12_SAMEDI,République,0,0,0,0,0,1,0,weekend
1460548,12_SAMEDI-16735903,1 days 01:24:31,1 days 01:24:31,6-1173,2,NaN,0,0,NaN,1,...,expl-12_SAMEDI,Pont de Strasbourg,0,0,0,0,0,1,0,weekend
1460549,12_SAMEDI-16735903,1 days 01:29:00,1 days 01:29:00,6-1610,3,NaN,0,0,NaN,1,...,expl-12_SAMEDI,Tournebride,0,0,0,0,0,1,0,weekend


In [ ]:
# Ajouter tranche horaire
def time_slot(td):
    h = td.seconds // 3600  # récupérer l'heure
    if 5 <= h < 7:
        return 'early birds'
    elif 7 <= h < 10:
        return 'morning commute'
    elif 10 <= h < 12:
        return 'late morning'
    elif 12 <= h < 14:
        return 'lunch time'
    elif 14 <= h < 17:
        return 'afternoon'
    elif 17 <= h < 20:
        return 'evening commute'
    elif 20 <= h < 22:
        return 'evening'
    else:
        return 'night'

full_df['time_slot'] = full_df['arrival_time'].apply(time_slot)
full_df.head()

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint,...,stop_name,monday,tuesday,wednesday,thursday,friday,saturday,sunday,day_type,time_slot
0,00000000001,0 days 05:16:00,0 days 05:16:00,6-5001,1,NaN,0,1,92.210907,1,...,J.F. Kennedy,1,0,0,0,0,0,0,weekday,early birds
1,00000000001,0 days 05:16:59,0 days 05:16:59,6-5002,2,NaN,0,0,616.730286,1,...,Villejean-Université,1,0,0,0,0,0,0,weekday,early birds
2,00000000001,0 days 05:18:29,0 days 05:18:29,6-5003,3,NaN,0,0,1414.565674,1,...,Pontchaillou,1,0,0,0,0,0,0,weekday,early birds
3,00000000001,0 days 05:19:39,0 days 05:19:39,6-5004,4,NaN,0,0,2050.249023,1,...,Anatole France,1,0,0,0,0,0,0,weekday,early birds
4,00000000001,0 days 05:20:47,0 days 05:20:47,6-5005,5,NaN,0,0,2726.148193,1,...,Sainte-Anne,1,0,0,0,0,0,0,weekday,early birds


In [ ]:
# Calculs agrégés par route, arrêt, day_type et service_id (jour)
daily_counts = full_df.groupby(['route_id', 'stop_id', 'stop_name', 'day_type', 'service_id']).agg(
    first_passage=('arrival_time', 'min'),
    last_passage=('arrival_time', 'max'),
    total_passages=('arrival_time', 'count')
).reset_index()

display(daily_counts)

,route_id,stop_id,stop_name,day_type,service_id,first_passage,last_passage,total_passages
0,6-0001,6-1040,Donelière,weekday,expl-12_JEUDI,0 days 05:27:49,1 days 01:09:49,102
1,6-0001,6-1040,Donelière,weekday,expl-12_LUNDI,0 days 05:27:49,1 days 00:10:49,101
2,6-0001,6-1040,Donelière,weekday,expl-12_MARDI,0 days 05:27:49,1 days 00:10:49,101
3,6-0001,6-1040,Donelière,weekday,expl-12_MERCREDI,0 days 05:27:49,1 days 00:10:49,99
4,6-0001,6-1040,Donelière,weekday,expl-12_VENDREDI,0 days 05:27:49,1 days 01:09:49,102
...,...,...,...,...,...,...,...,...
46396,6-1002,6-5079,Cesson - Viasilva,weekday,prog-00000000029,0 days 05:32:23,1 days 00:44:23,360
46397,6-1002,6-5079,Cesson - Viasilva,weekday,prog-00000000030,0 days 05:32:23,1 days 00:44:23,360
46398,6-1002,6-5079,Cesson - Viasilva,weekday,prog-00000000031,0 days 05:32:23,1 days 00:44:23,363
46399,6-1002,6-5079,Cesson - Viasilva,weekend,prog-00000000026,0 days 07:37:23,1 days 00:44:23,207


In [ ]:
# Calculer moyenne par jour pour le day_type 
agg = daily_counts.groupby(['route_id', 'stop_id', 'stop_name', 'day_type']).agg(
    first_passage=('first_passage', 'min'),   # premier passage du jour
    last_passage=('last_passage', 'max'),     # dernier passage du jour
    avg_passage_per_day=('total_passages', 'mean')  # moyenne passages par jour
).reset_index()

display(agg)

,route_id,stop_id,stop_name,day_type,first_passage,last_passage,avg_passage_per_day
0,6-0001,6-1040,Donelière,weekday,0 days 05:27:49,1 days 01:10:00,100.2
1,6-0001,6-1040,Donelière,weekend,0 days 06:08:00,1 days 01:10:00,52.5
2,6-0001,6-1041,Trois Croix,weekday,0 days 05:56:31,1 days 01:47:01,100.9
3,6-0001,6-1041,Trois Croix,weekend,0 days 06:36:31,1 days 01:47:01,52.5
4,6-0001,6-1042,Cimetière Nord,weekday,0 days 05:29:00,1 days 01:11:29,100.2
...,...,...,...,...,...,...,...
7179,6-1002,6-5077,Beaulieu - Université,weekend,0 days 05:28:53,1 days 01:40:53,252.0
7180,6-1002,6-5078,Atalante,weekday,0 days 05:30:41,1 days 01:42:41,364.6
7181,6-1002,6-5078,Atalante,weekend,0 days 05:30:41,1 days 01:42:41,252.0
7182,6-1002,6-5079,Cesson - Viasilva,weekday,0 days 05:32:23,1 days 01:44:23,364.6


In [ ]:
# Compter passages par tranche horaire et service_id
slot_counts = full_df.groupby(['route_id','stop_id','stop_name','day_type','service_id','time_slot']).size().reset_index(name='passages_in_slot')

display(slot_counts)

,route_id,stop_id,stop_name,day_type,service_id,time_slot,passages_in_slot
0,6-0001,6-1040,Donelière,weekday,expl-12_JEUDI,afternoon,20
1,6-0001,6-1040,Donelière,weekday,expl-12_JEUDI,early birds,5
2,6-0001,6-1040,Donelière,weekday,expl-12_JEUDI,evening,6
3,6-0001,6-1040,Donelière,weekday,expl-12_JEUDI,evening commute,22
4,6-0001,6-1040,Donelière,weekday,expl-12_JEUDI,late morning,12
...,...,...,...,...,...,...,...
207254,6-1002,6-5079,Cesson - Viasilva,weekend,prog-00000000032,evening commute,62
207255,6-1002,6-5079,Cesson - Viasilva,weekend,prog-00000000032,late morning,31
207256,6-1002,6-5079,Cesson - Viasilva,weekend,prog-00000000032,lunch time,31
207257,6-1002,6-5079,Cesson - Viasilva,weekend,prog-00000000032,morning commute,33


In [14]:
# Moyenne par jour pour chaque tranche
slot_avg = slot_counts.groupby(['route_id','stop_id','stop_name','day_type','time_slot'])['passages_in_slot'].mean().reset_index()

display(slot_avg)

,route_id,stop_id,stop_name,day_type,time_slot,passages_in_slot
0,6-0001,6-1040,Donelière,weekday,afternoon,19.4
1,6-0001,6-1040,Donelière,weekday,early birds,5.0
2,6-0001,6-1040,Donelière,weekday,evening,6.0
3,6-0001,6-1040,Donelière,weekday,evening commute,21.4
4,6-0001,6-1040,Donelière,weekday,late morning,12.0
...,...,...,...,...,...,...
33628,6-1002,6-5079,Cesson - Viasilva,weekend,evening commute,57.5
33629,6-1002,6-5079,Cesson - Viasilva,weekend,late morning,26.0
33630,6-1002,6-5079,Cesson - Viasilva,weekend,lunch time,25.5
33631,6-1002,6-5079,Cesson - Viasilva,weekend,morning commute,28.5


In [15]:
# Pivot pour avoir chaque tranche en colonne
slot_pivot = slot_avg.pivot_table(
    index=['route_id','stop_id','stop_name','day_type'],
    columns='time_slot',
    values='passages_in_slot',
    fill_value=0
).reset_index()

display(slot_pivot)

time_slot,route_id,stop_id,stop_name,day_type,afternoon,early birds,evening,evening commute,late morning,lunch time,morning commute,night
0,6-0001,6-1040,Donelière,weekday,19.4,5.0,6.0,21.4,12.0,12.0,20.0,4.4
1,6-0001,6-1040,Donelière,weekend,12.0,2.0,5.5,11.0,6.0,6.0,6.5,4.5
2,6-0001,6-1041,Trois Croix,weekday,18.0,3.0,7.0,23.0,12.0,12.0,20.5,5.4
3,6-0001,6-1041,Trois Croix,weekend,12.0,1.0,5.5,12.0,5.5,5.5,6.0,5.5
4,6-0001,6-1042,Cimetière Nord,weekday,19.4,5.0,6.0,21.4,12.0,12.0,20.0,4.4
...,...,...,...,...,...,...,...,...,...,...,...,...
7179,6-1002,6-5077,Beaulieu - Université,weekend,50.0,16.0,24.0,57.5,25.5,25.5,29.0,32.5
7180,6-1002,6-5078,Atalante,weekday,60.0,15.0,26.0,75.0,34.0,46.6,76.0,32.0
7181,6-1002,6-5078,Atalante,weekend,49.5,15.0,24.0,57.5,25.0,26.0,29.5,33.0
7182,6-1002,6-5079,Cesson - Viasilva,weekday,60.0,15.0,26.0,76.0,33.0,46.6,76.0,32.0


In [ ]:
# Fusionner avec la table agrégée 
final_table = agg.merge(slot_pivot, on=['route_id','stop_id','stop_name','day_type'], how='left')

# Formatage final des heures
final_table['first_passage'] = final_table['first_passage'].apply(lambda x: str(x))
final_table['last_passage'] = final_table['last_passage'].apply(lambda x: str(x))

final_table

,route_id,stop_id,stop_name,day_type,first_passage,last_passage,avg_passage_per_day,afternoon,early birds,evening,evening commute,late morning,lunch time,morning commute,night
0,6-0001,6-1040,Donelière,weekday,0 days 05:27:49,1 days 01:10:00,100.2,19.4,5.0,6.0,21.4,12.0,12.0,20.0,4.4
1,6-0001,6-1040,Donelière,weekend,0 days 06:08:00,1 days 01:10:00,52.5,12.0,2.0,5.5,11.0,6.0,6.0,6.5,4.5
2,6-0001,6-1041,Trois Croix,weekday,0 days 05:56:31,1 days 01:47:01,100.9,18.0,3.0,7.0,23.0,12.0,12.0,20.5,5.4
3,6-0001,6-1041,Trois Croix,weekend,0 days 06:36:31,1 days 01:47:01,52.5,12.0,1.0,5.5,12.0,5.5,5.5,6.0,5.5
4,6-0001,6-1042,Cimetière Nord,weekday,0 days 05:29:00,1 days 01:11:29,100.2,19.4,5.0,6.0,21.4,12.0,12.0,20.0,4.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7179,6-1002,6-5077,Beaulieu - Université,weekend,0 days 05:28:53,1 days 01:40:53,252.0,50.0,16.0,24.0,57.5,25.5,25.5,29.0,32.5
7180,6-1002,6-5078,Atalante,weekday,0 days 05:30:41,1 days 01:42:41,364.6,60.0,15.0,26.0,75.0,34.0,46.6,76.0,32.0
7181,6-1002,6-5078,Atalante,weekend,0 days 05:30:41,1 days 01:42:41,252.0,49.5,15.0,24.0,57.5,25.0,26.0,29.5,33.0
7182,6-1002,6-5079,Cesson - Viasilva,weekday,0 days 05:32:23,1 days 01:44:23,364.6,60.0,15.0,26.0,76.0,33.0,46.6,76.0,32.0


In [ ]:
# Compter passages par ligne, arrêt, jour (service_id)
daily_counts = full_df.groupby(['route_id', 'stop_id', 'stop_name', 'day_type', 'service_id']).agg(
    first_passage=('arrival_time', 'min'),
    last_passage=('arrival_time', 'max'),
    total_passages=('arrival_time', 'count')
).reset_index()

# Moyenne par jour pour chaque ligne/arrêt 
agg = daily_counts.groupby(['route_id', 'stop_id', 'stop_name', 'day_type']).agg(
    first_passage=('first_passage', 'min'),
    last_passage=('last_passage', 'max'),
    avg_passage_per_day=('total_passages', 'mean')
).reset_index()

# Compter passages par tranche horaire et service_id 
slot_counts = full_df.groupby(['route_id','stop_id','stop_name','day_type','service_id','time_slot']).size().reset_index(name='passages_in_slot')

# Moyenne par jour pour chaque tranche
slot_avg = slot_counts.groupby(['route_id','stop_id','stop_name','day_type','time_slot'])['passages_in_slot'].mean().reset_index()

# Pivot pour avoir chaque tranche horaire en colonne
slot_pivot = slot_avg.pivot_table(
    index=['route_id','stop_id','stop_name','day_type'],
    columns='time_slot',
    values='passages_in_slot',
    fill_value=0
).reset_index()

# Fusionner avec la table agrégée
final_table = agg.merge(slot_pivot, on=['route_id','stop_id','stop_name','day_type'], how='left')

# Formatage final des heures
final_table['first_passage'] = final_table['first_passage'].apply(lambda x: str(x))
final_table['last_passage'] = final_table['last_passage'].apply(lambda x: str(x))

final_table

,route_id,stop_id,stop_name,day_type,first_passage,last_passage,avg_passage_per_day,afternoon,early birds,evening,evening commute,late morning,lunch time,morning commute,night
0,6-0001,6-1040,Donelière,weekday,0 days 05:27:49,1 days 01:10:00,100.2,19.4,5.0,6.0,21.4,12.0,12.0,20.0,4.4
1,6-0001,6-1040,Donelière,weekend,0 days 06:08:00,1 days 01:10:00,52.5,12.0,2.0,5.5,11.0,6.0,6.0,6.5,4.5
2,6-0001,6-1041,Trois Croix,weekday,0 days 05:56:31,1 days 01:47:01,100.9,18.0,3.0,7.0,23.0,12.0,12.0,20.5,5.4
3,6-0001,6-1041,Trois Croix,weekend,0 days 06:36:31,1 days 01:47:01,52.5,12.0,1.0,5.5,12.0,5.5,5.5,6.0,5.5
4,6-0001,6-1042,Cimetière Nord,weekday,0 days 05:29:00,1 days 01:11:29,100.2,19.4,5.0,6.0,21.4,12.0,12.0,20.0,4.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7179,6-1002,6-5077,Beaulieu - Université,weekend,0 days 05:28:53,1 days 01:40:53,252.0,50.0,16.0,24.0,57.5,25.5,25.5,29.0,32.5
7180,6-1002,6-5078,Atalante,weekday,0 days 05:30:41,1 days 01:42:41,364.6,60.0,15.0,26.0,75.0,34.0,46.6,76.0,32.0
7181,6-1002,6-5078,Atalante,weekend,0 days 05:30:41,1 days 01:42:41,252.0,49.5,15.0,24.0,57.5,25.0,26.0,29.5,33.0
7182,6-1002,6-5079,Cesson - Viasilva,weekday,0 days 05:32:23,1 days 01:44:23,364.6,60.0,15.0,26.0,76.0,33.0,46.6,76.0,32.0


In [18]:
final_table['id_ligne'] = final_table['route_id'].apply(lambda x: x.split('-')[1] if '-' in x else x)
final_table['id_arret'] = final_table['stop_id'].apply(lambda x: x.split('-')[1] if '-' in x else x)

final_table

,route_id,stop_id,stop_name,day_type,first_passage,last_passage,avg_passage_per_day,afternoon,early birds,evening,evening commute,late morning,lunch time,morning commute,night,id_ligne,id_arret
0,6-0001,6-1040,Donelière,weekday,0 days 05:27:49,1 days 01:10:00,100.2,19.4,5.0,6.0,21.4,12.0,12.0,20.0,4.4,0001,1040
1,6-0001,6-1040,Donelière,weekend,0 days 06:08:00,1 days 01:10:00,52.5,12.0,2.0,5.5,11.0,6.0,6.0,6.5,4.5,0001,1040
2,6-0001,6-1041,Trois Croix,weekday,0 days 05:56:31,1 days 01:47:01,100.9,18.0,3.0,7.0,23.0,12.0,12.0,20.5,5.4,0001,1041
3,6-0001,6-1041,Trois Croix,weekend,0 days 06:36:31,1 days 01:47:01,52.5,12.0,1.0,5.5,12.0,5.5,5.5,6.0,5.5,0001,1041
4,6-0001,6-1042,Cimetière Nord,weekday,0 days 05:29:00,1 days 01:11:29,100.2,19.4,5.0,6.0,21.4,12.0,12.0,20.0,4.4,0001,1042
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7179,6-1002,6-5077,Beaulieu - Université,weekend,0 days 05:28:53,1 days 01:40:53,252.0,50.0,16.0,24.0,57.5,25.5,25.5,29.0,32.5,1002,5077
7180,6-1002,6-5078,Atalante,weekday,0 days 05:30:41,1 days 01:42:41,364.6,60.0,15.0,26.0,75.0,34.0,46.6,76.0,32.0,1002,5078
7181,6-1002,6-5078,Atalante,weekend,0 days 05:30:41,1 days 01:42:41,252.0,49.5,15.0,24.0,57.5,25.0,26.0,29.5,33.0,1002,5078
7182,6-1002,6-5079,Cesson - Viasilva,weekday,0 days 05:32:23,1 days 01:44:23,364.6,60.0,15.0,26.0,76.0,33.0,46.6,76.0,32.0,1002,5079


In [19]:
# Charger la table des arrêts
arrets_df = pd.read_csv(r'C:\Users\justi\OneDrive\Je-deviens-Data-Analyst\JEDHA\2_Fullstack\11_final_project\1 - DATA\2 - DATA SILVER\stg_star_bus_arrets.csv', sep=';', dtype={'id_arret': str})

arrets_df.head()

,id_arret,nom_arret,code_geo,latitude,longitude
0,2002,Cesson Hôpital Privé,35051,48.130225,-1.627669
1,2013,Vilaine,35051,48.116363,-1.607332
2,2044,Belle Fontaine,35051,48.127353,-1.624328
3,2045,Atalante,35051,48.127339,-1.628368
4,2050,Cesson Collège,35051,48.120327,-1.599292


In [20]:
# Filtre : garder uniquement les arrêts présents dans stg_star_bus_arrets
mask = final_table['id_arret'].isin(arrets_df['id_arret'])

# Appliquer le filtre
final_table_mvp = final_table[mask].copy()

display(final_table_mvp.head())
print(f"Nombre de lignes après filtrage : {len(final_table_mvp)}")

,route_id,stop_id,stop_name,day_type,first_passage,last_passage,avg_passage_per_day,afternoon,early birds,evening,evening commute,late morning,lunch time,morning commute,night,id_ligne,id_arret
94,6-0001,6-2804,Ricoquais,weekday,0 days 05:18:00,1 days 01:02:00,99.4,19.0,6.0,5.0,20.0,12.0,12.0,21.0,4.4,0001,2804
95,6-0001,6-2804,Ricoquais,weekend,0 days 05:58:00,1 days 01:02:00,52.5,12.0,2.0,5.0,11.0,5.5,6.5,7.0,4.5,0001,2804
96,6-0001,6-2805,Camus,weekday,0 days 05:19:00,1 days 01:02:21,99.4,19.0,6.0,5.0,20.0,12.0,12.0,21.0,4.4,0001,2805
97,6-0001,6-2805,Camus,weekend,0 days 05:59:00,1 days 01:02:21,52.5,12.0,2.0,5.0,11.0,5.5,6.5,7.0,4.5,0001,2805
98,6-0001,6-2806,Vivier Louis,weekday,0 days 05:20:00,1 days 01:03:00,99.4,19.0,6.0,6.0,19.0,12.0,12.0,21.0,4.4,0001,2806


Nombre de lignes après filtrage : 4775


In [21]:
final_table_mvp.isna().sum()

route_id               0
stop_id                0
stop_name              0
day_type               0
first_passage          0
last_passage           0
avg_passage_per_day    0
afternoon              0
early birds            0
evening                0
evening commute        0
late morning           0
lunch time             0
morning commute        0
night                  0
id_ligne               0
id_arret               0
dtype: int64

In [22]:
final_table_mvp.describe(include="all")

,route_id,stop_id,stop_name,day_type,first_passage,last_passage,avg_passage_per_day,afternoon,early birds,evening,evening commute,late morning,lunch time,morning commute,night,id_ligne,id_arret
count,4775,4775,4775,4775,4775,4775,4775.000000,4775.000000,4775.000000,4775.000000,4775.000000,4775.000000,4775.000000,4775.000000,4775.000000,4775,4775
unique,147,1150,593,2,1531,1497,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,147,1150
top,6-0806,6-2366,Saint-Jacques - Gaîté,weekday,0 days 07:47:00,0 days 17:10:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0806,2366
freq,253,14,29,3108,45,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,253,14
mean,NaN,NaN,NaN,NaN,NaN,NaN,16.409571,3.347215,0.529756,1.298953,4.187675,1.636178,2.003780,3.480112,0.682220,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,21.237608,4.377865,1.132013,1.950288,4.872736,2.502270,2.473776,4.369217,2.028323,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,1.800000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,6.000000,1.000000,0.000000,0.000000,2.000000,0.000000,1.000000,2.000000,0.000000,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,24.750000,6.000000,1.000000,2.000000,6.600000,3.000000,3.000000,6.000000,0.000000,NaN,NaN


In [27]:
final_table_mvp.dtypes

route_id                object
stop_id                 object
stop_name               object
day_type                object
first_passage           object
last_passage            object
avg_passage_per_day    float64
afternoon              float64
early birds            float64
evening                float64
evening commute        float64
late morning           float64
lunch time             float64
morning commute        float64
night                  float64
id_ligne                object
id_arret                object
dtype: object

In [28]:
final_table_mvp.columns

Index(['route_id', 'stop_id', 'stop_name', 'day_type', 'first_passage',
       'last_passage', 'avg_passage_per_day', 'afternoon', 'early birds',
       'evening', 'evening commute', 'late morning', 'lunch time',
       'morning commute', 'night', 'id_ligne', 'id_arret'],
      dtype='object')

In [29]:
cols= ['id_ligne', 'id_arret', 'stop_name', 'day_type', 'first_passage', 'last_passage', 'avg_passage_per_day', 'early birds', 'morning commute', 'late morning', 'lunch time','afternoon', 'evening commute', 'evening', 'night']
gtfs_mvp= final_table_mvp[cols]
gtfs_mvp

,id_ligne,id_arret,stop_name,day_type,first_passage,last_passage,avg_passage_per_day,early birds,morning commute,late morning,lunch time,afternoon,evening commute,evening,night
94,0001,2804,Ricoquais,weekday,0 days 05:18:00,1 days 01:02:00,99.40,6.00,21.00,12.0,12.0,19.0,20.0,5.0,4.40
95,0001,2804,Ricoquais,weekend,0 days 05:58:00,1 days 01:02:00,52.50,2.00,7.00,5.5,6.5,12.0,11.0,5.0,4.50
96,0001,2805,Camus,weekday,0 days 05:19:00,1 days 01:02:21,99.40,6.00,21.00,12.0,12.0,19.0,20.0,5.0,4.40
97,0001,2805,Camus,weekend,0 days 05:59:00,1 days 01:02:21,52.50,2.00,7.00,5.5,6.5,12.0,11.0,5.0,4.50
98,0001,2806,Vivier Louis,weekday,0 days 05:20:00,1 days 01:03:00,99.40,6.00,21.00,12.0,12.0,19.0,19.0,6.0,4.40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7055,0810,2266,Parc Expo,weekday,0 days 19:17:00,1 days 07:30:00,92.75,9.50,2.50,0.0,0.0,0.0,6.0,30.5,68.50
7056,0810,2266,Parc Expo,weekend,0 days 04:24:00,1 days 07:30:00,90.00,25.25,6.25,0.0,0.0,0.0,4.0,37.0,48.25
7057,0810,2279,Saint-Jacques - Gaîté,weekday,0 days 19:34:00,1 days 07:43:00,46.00,4.50,1.50,0.0,0.0,0.0,2.0,14.5,34.75
7058,0810,2279,Saint-Jacques - Gaîté,weekend,0 days 04:37:00,1 days 07:43:00,46.50,12.75,4.75,0.0,0.0,0.0,1.0,17.0,24.50


In [32]:
# Ajout de l'amplitude horaire

gtfs_mvp['first_passage'] = pd.to_timedelta(gtfs_mvp['first_passage'], errors='coerce')
gtfs_mvp['last_passage'] = pd.to_timedelta(gtfs_mvp['last_passage'], errors='coerce')

gtfs_mvp['amplitude_horaire']= gtfs_mvp['last_passage'] - gtfs_mvp['first_passage']

gtfs_mvp

C:\Users\justi\AppData\Local\Temp\ipykernel_22776\824055593.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gtfs_mvp['first_passage'] = pd.to_timedelta(gtfs_mvp['first_passage'], errors='coerce')
C:\Users\justi\AppData\Local\Temp\ipykernel_22776\824055593.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gtfs_mvp['last_passage'] = pd.to_timedelta(gtfs_mvp['last_passage'], errors='coerce')
C:\Users\justi\AppData\Local\Temp\ipykernel_22776\824055593.py:6: SettingWithCopyWarning: 
A value is trying to be

,id_ligne,id_arret,stop_name,day_type,first_passage,last_passage,avg_passage_per_day,early birds,morning commute,late morning,lunch time,afternoon,evening commute,evening,night,amplitude_horaire
94,0001,2804,Ricoquais,weekday,0 days 05:18:00,1 days 01:02:00,99.40,6.00,21.00,12.0,12.0,19.0,20.0,5.0,4.40,0 days 19:44:00
95,0001,2804,Ricoquais,weekend,0 days 05:58:00,1 days 01:02:00,52.50,2.00,7.00,5.5,6.5,12.0,11.0,5.0,4.50,0 days 19:04:00
96,0001,2805,Camus,weekday,0 days 05:19:00,1 days 01:02:21,99.40,6.00,21.00,12.0,12.0,19.0,20.0,5.0,4.40,0 days 19:43:21
97,0001,2805,Camus,weekend,0 days 05:59:00,1 days 01:02:21,52.50,2.00,7.00,5.5,6.5,12.0,11.0,5.0,4.50,0 days 19:03:21
98,0001,2806,Vivier Louis,weekday,0 days 05:20:00,1 days 01:03:00,99.40,6.00,21.00,12.0,12.0,19.0,19.0,6.0,4.40,0 days 19:43:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7055,0810,2266,Parc Expo,weekday,0 days 19:17:00,1 days 07:30:00,92.75,9.50,2.50,0.0,0.0,0.0,6.0,30.5,68.50,0 days 12:13:00
7056,0810,2266,Parc Expo,weekend,0 days 04:24:00,1 days 07:30:00,90.00,25.25,6.25,0.0,0.0,0.0,4.0,37.0,48.25,1 days 03:06:00
7057,0810,2279,Saint-Jacques - Gaîté,weekday,0 days 19:34:00,1 days 07:43:00,46.00,4.50,1.50,0.0,0.0,0.0,2.0,14.5,34.75,0 days 12:09:00
7058,0810,2279,Saint-Jacques - Gaîté,weekend,0 days 04:37:00,1 days 07:43:00,46.50,12.75,4.75,0.0,0.0,0.0,1.0,17.0,24.50,1 days 03:06:00


In [34]:
# Moyenne de passages par heure selon amplitude horaire

gtfs_mvp['amplitude_horaire_heures'] = gtfs_mvp['amplitude_horaire'] / pd.Timedelta(hours=1)

gtfs_mvp['avg_passage_per_hour'] = (
    gtfs_mvp['avg_passage_per_day'] / gtfs_mvp['amplitude_horaire_heures']
)

gtfs_mvp

,id_ligne,id_arret,stop_name,day_type,first_passage,last_passage,avg_passage_per_day,early birds,morning commute,late morning,lunch time,afternoon,evening commute,evening,night,amplitude_horaire,amplitude_horaire_heures,avg_passage_per_hour
94,0001,2804,Ricoquais,weekday,0 days 05:18:00,1 days 01:02:00,99.40,6.00,21.00,12.0,12.0,19.0,20.0,5.0,4.40,0 days 19:44:00,19.733333,5.037162
95,0001,2804,Ricoquais,weekend,0 days 05:58:00,1 days 01:02:00,52.50,2.00,7.00,5.5,6.5,12.0,11.0,5.0,4.50,0 days 19:04:00,19.066667,2.753497
96,0001,2805,Camus,weekday,0 days 05:19:00,1 days 01:02:21,99.40,6.00,21.00,12.0,12.0,19.0,20.0,5.0,4.40,0 days 19:43:21,19.722500,5.039929
97,0001,2805,Camus,weekend,0 days 05:59:00,1 days 01:02:21,52.50,2.00,7.00,5.5,6.5,12.0,11.0,5.0,4.50,0 days 19:03:21,19.055833,2.755062
98,0001,2806,Vivier Louis,weekday,0 days 05:20:00,1 days 01:03:00,99.40,6.00,21.00,12.0,12.0,19.0,19.0,6.0,4.40,0 days 19:43:00,19.716667,5.041420
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7055,0810,2266,Parc Expo,weekday,0 days 19:17:00,1 days 07:30:00,92.75,9.50,2.50,0.0,0.0,0.0,6.0,30.5,68.50,0 days 12:13:00,12.216667,7.592087
7056,0810,2266,Parc Expo,weekend,0 days 04:24:00,1 days 07:30:00,90.00,25.25,6.25,0.0,0.0,0.0,4.0,37.0,48.25,1 days 03:06:00,27.100000,3.321033
7057,0810,2279,Saint-Jacques - Gaîté,weekday,0 days 19:34:00,1 days 07:43:00,46.00,4.50,1.50,0.0,0.0,0.0,2.0,14.5,34.75,0 days 12:09:00,12.150000,3.786008
7058,0810,2279,Saint-Jacques - Gaîté,weekend,0 days 04:37:00,1 days 07:43:00,46.50,12.75,4.75,0.0,0.0,0.0,1.0,17.0,24.50,1 days 03:06:00,27.100000,1.715867


In [35]:
gtfs_mvp.to_csv("gtfs_mvp.csv", sep=";", index=False)